In [ ]:
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
# Define paths to the project datasets and folders
MASTER_PATH = Path("../data/processed/crime_2010_2024.parquet")
PARTITIONED_PATH = Path("../data/dashboard/crime_data")
SUMMARY_PATH = Path("../data/dashboard/summaries")

# Print the full paths to verify the locations
print("Master dataset:", MASTER_PATH.resolve())
print("Partitioned dataset:", PARTITIONED_PATH.resolve())
print("Summary directory:", SUMMARY_PATH.resolve())

In [ ]:
# Load the cleaned master dataset from Notebook 1
df = pd.read_parquet(MASTER_PATH)

# Inspect the dataset's dimensions and column names
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

In [ ]:
# Ensure every record has a valid year before partitioning
assert df["year"].notna().all(), "Missing values found in year."

# Check the year range and available years
print("Year range:", df["year"].min(), "to", df["year"].max())
print("Unique years:", sorted(df["year"].unique().tolist()))

In [ ]:
# Remove old dashboard partitions to prevent stale data
if PARTITIONED_PATH.exists():
    shutil.rmtree(PARTITIONED_PATH)

# Create a fresh directory for the new partitions
PARTITIONED_PATH.mkdir(parents=True, exist_ok=True)

print("Dashboard partition directory is ready.")

In [ ]:
# Save the dataset as separate Parquet files for each year
# This allows the dashboard to load only the years it needs

df.to_parquet(PARTITIONED_PATH,engine="pyarrow",partition_cols=["year"],index=False,)

print("Partitioned Parquet dataset created successfully.")

In [ ]:
# List the generated year-based partition folders
year_partitions = sorted(
    path.name
    for path in PARTITIONED_PATH.iterdir()
    if path.is_dir() and path.name.startswith("year=")
)

print("Partitions created:")
for partition in year_partitions:
    print("-", partition)

# Verify that every year in the dataset has a partition
expected_partitions = {
    f"year={int(year)}" for year in df["year"].unique()
}

assert set(year_partitions) == expected_partitions, "Partition validation failed."

print("\nPartition validation passed.")

In [ ]:
# Test whether Parquet filtering loads data for only the requested year
test_year = 2022

df_2022 = pd.read_parquet(PARTITIONED_PATH,filters=[("year", "==", test_year)],)

# Check the number of rows and the year returned
print(f"Rows loaded for {test_year}:", len(df_2022))
print("Years present:", df_2022["year"].unique())

# Confirm that the filtered data contains only the requested year
assert df_2022["year"].nunique() == 1
assert int(df_2022["year"].iloc[0]) == test_year

print("\nFiltered Parquet read passed.")

In [ ]:
# Count records across all year partitions to verify data integrity
partitioned_total = sum(
    len(pd.read_parquet(PARTITIONED_PATH,filters=[("year", "==", int(year))],columns=["DR_NO"],))
    for year in df["year"].unique()
)

# Compare the master and partitioned dataset sizes
print("Master rows:", len(df))
print("Partitioned rows:", partitioned_total)

assert partitioned_total == len(df), "Row-count validation failed."

print("\nRow-count validation passed.")

In [ ]:
# Create a summary of incident counts for each year
yearly_crime = (df.groupby("year").size().reset_index(name="incident_count")
                .sort_values("year").reset_index(drop=True)
)

# Display the summary table and its row count
display(yearly_crime)
print("Rows:", len(yearly_crime))

In [ ]:
# Create monthly incident counts for each year
monthly_crime = (df.groupby(["year", "month", "month_name"], as_index=False).size()
    .rename(columns={"size": "incident_count"}).sort_values(["year", "month"]).reset_index(drop=True)
)

# Preview the results and check the number of rows
display(monthly_crime.head(20))
print("Rows:", len(monthly_crime))

In [ ]:
# Count incidents by year and crime type
crime_by_year = (df.dropna(subset=["Crm Cd Desc"]).groupby(["year", "Crm Cd Desc"], as_index=False)
    .size().rename(columns={"size": "incident_count"})
    .sort_values(["year", "incident_count"], ascending=[True, False]).reset_index(drop=True)
)

# Preview the results and check the number of rows
display(crime_by_year.head(20))
print("Rows:", len(crime_by_year))

In [ ]:
# Count incidents by LAPD area and year
area_by_year = (df.dropna(subset=["AREA NAME"]).groupby(["year", "AREA NAME"], as_index=False)
    .size().rename(columns={"size": "incident_count"})
    .sort_values(["year", "incident_count"], ascending=[True, False]).reset_index(drop=True)
)

# Preview the results and check the number of rows
display(area_by_year.head(20))
print("Rows:", len(area_by_year))

In [ ]:
# Count incidents by premise type and year
premise_by_year = (df.dropna(subset=["Premis Desc"]).groupby(["year", "Premis Desc"], as_index=False)
    .size().rename(columns={"size": "incident_count"})
    .sort_values(["year", "incident_count"], ascending=[True, False]).reset_index(drop=True)
)

# Preview the results and check the number of rows
display(premise_by_year.head(20))
print("Rows:", len(premise_by_year))

In [ ]:
# Create the directory for dashboard summary tables
SUMMARY_PATH.mkdir(parents=True, exist_ok=True)

# Save each summary as a Parquet file
summaries = {
    "yearly_crime": yearly_crime,
    "monthly_crime": monthly_crime,
    "crime_by_year": crime_by_year,
    "area_by_year": area_by_year,
    "premise_by_year": premise_by_year,
}

for name, summary in summaries.items():
    summary.to_parquet(SUMMARY_PATH / f"{name}.parquet",engine="pyarrow",index=False,)

print("Summary Parquet files created successfully.")

In [ ]:
# Verify that each saved summary can be reloaded correctly
summaries = {
    "yearly_crime": yearly_crime,
    "monthly_crime": monthly_crime,
    "crime_by_year": crime_by_year,
    "area_by_year": area_by_year,
    "premise_by_year": premise_by_year,
}

for name, original in summaries.items():
    check = pd.read_parquet(SUMMARY_PATH / f"{name}.parquet")

    # Check the saved table's dimensions and structure
    print(f"{name:20s} | Rows: {len(check):,} | Columns: {len(check.columns)}")

    assert len(check) == len(original), f"{name}: row count mismatch"
    assert check.columns.tolist() == original.columns.tolist(), (f"{name}: column mismatch")

print("\nAll summary Parquet files passed validation.")

In [ ]:
# Verify that yearly counts account for every master dataset record
summary_total = yearly_crime["incident_count"].sum()
master_total = len(df)

print(f"Master incident count: {master_total:,}")
print(f"Yearly summary count: {summary_total:,}")

assert summary_total == master_total, "Incident counts do not match."

print("\nYearly summary validation passed.")

In [ ]:
# Define time-of-day categories based on the incident hour
time_bins = [-1, 5, 11, 17, 23]
time_labels = ["Night", "Morning", "Afternoon", "Evening"]

# Create a separate DataFrame for dashboard-specific features
df_dashboard = df.copy()

df_dashboard["time_of_day"] = pd.cut(df_dashboard["hour"],bins=time_bins,labels=time_labels,)

In [ ]:
# Count incidents by year, month, crime type, and LAPD area
crime_area_month = (
    df_dashboard
    .dropna(subset=["Crm Cd Desc", "AREA NAME"])
    .groupby(["year", "month", "month_name", "Crm Cd Desc", "AREA NAME"],as_index=False,)
    .size()
    .rename(columns={"size": "incident_count"})
    .sort_values(["year", "month", "Crm Cd Desc", "AREA NAME"])
    .reset_index(drop=True)
)

# Display summary information and preview the results
print("Crime + Area + Month summary:")
print(f"Rows: {len(crime_area_month):,}")

display(crime_area_month.head(20))

In [ ]:
# Count incidents by year, hour, crime type, and LAPD area
crime_area_hour = (
    df_dashboard
    .dropna(subset=["Crm Cd Desc", "AREA NAME", "hour"])
    .groupby(["year", "hour", "Crm Cd Desc", "AREA NAME"],as_index=False,)
    .size()
    .rename(columns={"size": "incident_count"})
    .sort_values(["year", "hour", "Crm Cd Desc", "AREA NAME"])
    .reset_index(drop=True)
)

# Display summary information and preview the results
print("Crime + Area + Hour summary:")
print(f"Rows: {len(crime_area_hour):,}")

display(crime_area_hour.head(20))

In [ ]:
# Count incidents by year, time of day, crime type, and LAPD area
crime_area_time_of_day = (
    df_dashboard
    .dropna(subset=["Crm Cd Desc", "AREA NAME", "time_of_day"])
    .groupby(["year", "time_of_day", "Crm Cd Desc", "AREA NAME"],observed=True,as_index=False,)
    .size()
    .rename(columns={"size": "incident_count"})
    .sort_values(["year", "time_of_day", "Crm Cd Desc", "AREA NAME"])
    .reset_index(drop=True)
)

# Display summary information and preview the results
print("Crime + Area + Time-of-day summary:")
print(f"Rows: {len(crime_area_time_of_day):,}")

display(crime_area_time_of_day.head(20))

In [ ]:
# Count incidents by year, weekday, hour, crime type, and LAPD area
crime_area_day_hour = (
    df_dashboard
    .dropna(subset=["Crm Cd Desc", "AREA NAME", "day_of_week", "day_name", "hour"])
    .groupby(["year", "day_of_week", "day_name", "hour", "Crm Cd Desc", "AREA NAME"],as_index=False,)
    .size()
    .rename(columns={"size": "incident_count"})
    .sort_values(["year", "day_of_week", "hour", "Crm Cd Desc", "AREA NAME"]
    ).reset_index(drop=True)
)

# Display summary information and preview the results
print("Crime + Area + Day + Hour summary:")
print(f"Rows: {len(crime_area_day_hour):,}")

display(crime_area_day_hour.head(20))

In [ ]:
# Save the Crime & Time summaries for the dashboard
crime_time_summaries = {
    "crime_area_month": crime_area_month,
    "crime_area_hour": crime_area_hour,
    "crime_area_time_of_day": crime_area_time_of_day,
    "crime_area_day_hour": crime_area_day_hour,
}

for name, summary in crime_time_summaries.items():
    summary.to_parquet(SUMMARY_PATH / f"{name}.parquet",engine="pyarrow",index=False,)

print("Crime & Time summary Parquet files created successfully.")

In [ ]:
# Verify that the saved Crime & Time summaries are readable and correctly structured
for name, original in crime_time_summaries.items():
    check = pd.read_parquet(SUMMARY_PATH / f"{name}.parquet")

    # Display the saved table's dimensions
    print(f"{name:25s} | Rows: {len(check):,} | Columns: {len(check.columns)}")

    # Check row counts and column names
    assert len(check) == len(original), f"{name}: row count mismatch"
    assert check.columns.tolist() == original.columns.tolist(), (f"{name}: column mismatch")

print("\nAll Crime & Time summary files passed validation.")

In [ ]:
# Select the columns needed for victim demographics analysis
victim_dashboard = df[
    ["year", "Crm Cd Desc", "Vict Age", "Vict Sex", "Vict Descent"]
].copy()

# Preserve missing demographic values for dashboard summaries
victim_dashboard[["Vict Sex", "Vict Descent"]] = (
    victim_dashboard[["Vict Sex", "Vict Descent"]].fillna("Missing")
)

In [ ]:
# Count incidents by year, crime type, and victim demographics
victim_profile = (
    victim_dashboard
    .groupby(["year", "Crm Cd Desc", "Vict Age", "Vict Sex", "Vict Descent"],dropna=False,as_index=False,)
    .size()
    .rename(columns={"size": "incident_count"})
    .sort_values(["year", "Crm Cd Desc", "Vict Age", "Vict Sex", "Vict Descent"]).reset_index(drop=True)
)

# Display summary information and preview the results
print("Victim profile summary:")
print(f"Rows: {len(victim_profile):,}")

display(victim_profile.head(20))

In [ ]:
# Count incidents by year and victim sex
victim_sex_by_year = (
    victim_dashboard
    .groupby(["year", "Vict Sex"], as_index=False, dropna=False)
    .size()
    .rename(columns={"size": "incident_count"})
    .sort_values(["year", "incident_count"], ascending=[True, False])
    .reset_index(drop=True)
)

# Preview the results and check the number of rows
display(victim_sex_by_year.head(20))
print(f"Rows: {len(victim_sex_by_year):,}")

In [ ]:
# Count incidents by year and victim descent
victim_descent_by_year = (
    victim_dashboard
    .groupby(["year", "Vict Descent"], as_index=False, dropna=False)
    .size()
    .rename(columns={"size": "incident_count"})
    .sort_values(["year", "incident_count"], ascending=[True, False])
    .reset_index(drop=True)
)

# Preview the results and check the number of rows
display(victim_descent_by_year.head(20))
print(f"Rows: {len(victim_descent_by_year):,}")

In [ ]:
# Count incidents by year and valid victim age
victim_age_by_year = (
    victim_dashboard
    .dropna(subset=["Vict Age"])
    .groupby(["year", "Vict Age"], as_index=False)
    .size()
    .rename(columns={"size": "incident_count"})
    .sort_values(["year", "Vict Age"])
    .reset_index(drop=True)
)

# Preview the results and check the number of rows
display(victim_age_by_year.head(20))
print(f"Rows: {len(victim_age_by_year):,}")

In [ ]:
# Save the Victims summary tables for the dashboard
victim_summaries = {
    "victim_profile": victim_profile,
    "victim_sex_by_year": victim_sex_by_year,
    "victim_descent_by_year": victim_descent_by_year,
    "victim_age_by_year": victim_age_by_year,
}

for name, summary in victim_summaries.items():
    summary.to_parquet(SUMMARY_PATH / f"{name}.parquet",engine="pyarrow",index=False,)

print("Victims summary Parquet files created successfully.")

In [ ]:
# Verify that the saved Victims summaries are readable and correctly structured
for name, original in victim_summaries.items():
    check = pd.read_parquet(SUMMARY_PATH / f"{name}.parquet")

    # Display the saved table's dimensions
    print(f"{name:25s} | Rows: {len(check):,} | Columns: {len(check.columns)}")

    # Check row counts and column names
    assert len(check) == len(original), f"{name}: row count mismatch"
    assert check.columns.tolist() == original.columns.tolist(), (f"{name}: column mismatch")

print("\nAll Victims summary files passed validation.")

In [ ]:
# Prepare the columns needed for area and crime comparisons
area_crime_dashboard = df[
    ["year", "Crm Cd Desc", "AREA NAME"]
].copy()

# Preserve missing categories in the dashboard summaries
area_crime_dashboard[["Crm Cd Desc", "AREA NAME"]] = (
    area_crime_dashboard[["Crm Cd Desc", "AREA NAME"]].fillna("Missing")
)

# Count incidents by year, crime type, and LAPD area
area_crime_by_year = (
    area_crime_dashboard
    .groupby(["year", "Crm Cd Desc", "AREA NAME"], as_index=False)
    .size()
    .rename(columns={"size": "incident_count"})
    .sort_values(["year", "incident_count"], ascending=[True, False])
    .reset_index(drop=True)
)

# Save the summary and preview the results
area_crime_by_year.to_parquet(
    SUMMARY_PATH / "area_crime_by_year.parquet",
    index=False,
)

print("Summary shape:", area_crime_by_year.shape)
display(area_crime_by_year.head())

In [ ]:
# Select the columns needed for premise and crime comparisons
premise_crime_dashboard = df[["year", "Crm Cd Desc", "Premis Desc"]]

# Preserve missing categories in the dashboard summaries
premise_crime_dashboard[["Crm Cd Desc", "Premis Desc"]] = (
    premise_crime_dashboard[["Crm Cd Desc", "Premis Desc"]].fillna("Missing")
)

# Count incidents by year, crime type, and premise
premise_crime_by_year = (
    premise_crime_dashboard
    .groupby(["year", "Crm Cd Desc", "Premis Desc"], as_index=False)
    .size()
    .rename(columns={"size": "incident_count"})
    .sort_values(["year", "incident_count"], ascending=[True, False])
    .reset_index(drop=True)
)

# Save the summary and preview the results
premise_crime_by_year.to_parquet(SUMMARY_PATH / "premise_crime_by_year.parquet",index=False)

print("Summary shape:", premise_crime_by_year.shape)
display(premise_crime_by_year.head())

In [ ]:
# Select the columns needed for reporting-lag analysis
lag_dashboard = df[["year", "reporting_lag_days"]].copy()

# Exclude missing and negative reporting lags
lag_valid = lag_dashboard.loc[
    lag_dashboard["reporting_lag_days"].notna()
    & (lag_dashboard["reporting_lag_days"] >= 0)
].copy()

# Convert valid reporting lags to whole days
lag_valid["reporting_lag_days"] = (
    lag_valid["reporting_lag_days"].round().astype(int)
)

# Summarize reporting delays by year
reporting_lag_by_year = (
    lag_valid
    .groupby("year", as_index=False)
    .agg(
        incident_count=("reporting_lag_days", "size"),
        median_lag_days=("reporting_lag_days", "median"),
        mean_lag_days=("reporting_lag_days", "mean"),
        max_lag_days=("reporting_lag_days", "max"),
    )
)

# Save yearly reporting-lag statistics
reporting_lag_by_year.to_parquet(
    SUMMARY_PATH / "reporting_lag_by_year.parquet",
    index=False,
)

# Count incidents for each reporting lag within each year
reporting_lag_distribution = (
    lag_valid
    .groupby(["year", "reporting_lag_days"], as_index=False)
    .size()
    .rename(columns={"size": "incident_count"})
    .sort_values(["year", "reporting_lag_days"])
    .reset_index(drop=True)
)

# Save the distribution for flexible dashboard analysis
reporting_lag_distribution.to_parquet(
    SUMMARY_PATH / "reporting_lag_distribution.parquet",
    index=False,
)

# Display summary sizes and preview the results
print("Yearly summary shape:", reporting_lag_by_year.shape)
print("Lag distribution shape:", reporting_lag_distribution.shape)

display(reporting_lag_by_year)
display(reporting_lag_distribution.head(20))

In [ ]:
# Select the columns needed for the dashboard map
geo_columns = [
    "year", "month", "LAT", "LON",
    "Crm Cd Desc", "AREA NAME", "Premis Desc",
]
geo_dashboard = df[geo_columns].copy()

# Keep incidents with valid coordinates inside the Los Angeles-area bounds
geo_valid = geo_dashboard.loc[
    geo_dashboard["LAT"].between(33.5, 34.5)
    & geo_dashboard["LON"].between(-119.0, -117.5)
].copy()

# Preserve missing category labels for dashboard filters
category_columns = ["Crm Cd Desc", "AREA NAME", "Premis Desc"]
geo_valid[category_columns] = geo_valid[category_columns].fillna("Missing")

# Sample up to 1,000 incidents per year for map performance
geo_sample = pd.concat(
    [
        group.sample(n=min(1000, len(group)), random_state=42)
        for _, group in geo_valid.groupby("year")
    ],
    ignore_index=True,
)

# Sort the sample and save it for the dashboard
geo_sample = geo_sample.sort_values(
    ["year", "LAT", "LON"]
).reset_index(drop=True)

geo_sample.to_parquet(
    SUMMARY_PATH / "geo_sample.parquet",
    index=False,
)

# Display the sample size and preview the results
print("Geographic sample shape:", geo_sample.shape)
display(geo_sample.head())

In [ ]:
# Verify that the Location & Reporting summary files can be read
summary_files = [
    "area_crime_by_year.parquet",
    "premise_crime_by_year.parquet",
    "reporting_lag_by_year.parquet",
    "reporting_lag_distribution.parquet",
    "geo_sample.parquet",
]

for filename in summary_files:
    check = pd.read_parquet(SUMMARY_PATH / filename)

    # Display each file's dimensions
    print(
        f"{filename:<40} "
        f"rows={len(check):,} | columns={len(check.columns)}"
    )

print("\nAll Location & Reporting summary files loaded successfully.")

In [ ]:
def categorize(text, rules, default, missing=None):
    text = text.fillna("").astype(str).str.upper().str.strip()

    conditions = []

    if missing:
        conditions.append(text.isin(["", "NONE"]))

    conditions += [
        text.apply(lambda x: any(term in x for term in terms))
        for _, terms in rules
    ]

    choices = ([missing] if missing else []) + [
        label for label, _ in rules
    ]

    return pd.Series(
        np.select(conditions, choices, default=default),
        index=text.index,
        dtype="string"
    )


# Crime categories
crime_rules = [
    ("Part 2 - Financial/White Collar", [
        "THEFT OF IDENTITY", "IDENTITY THEFT", "FRAUD", "FORGERY",
        "CREDIT CARD", "COUNTERFEIT", "EMBEZZ", "BAD CHECK",
        "FALSE PRETENSE", "DEFRAUD", "BUNCO"
    ]),
    ("Part 2 - Simple Assault", [
        "BATTERY - SIMPLE ASSAULT", "SIMPLE ASSAULT",
        "ASSAULT & BATTERY", "SIMPLE BATTERY"
    ]),
    ("Part 1 - Violent", [
        "MURDER", "HOMICIDE", "MANSLAUGHTER", "RAPE", "ROBBERY",
        "KIDNAPPING", "AGGRAVATED ASSAULT", "ASSAULT WITH DEADLY WEAPON",
        "ATTEMPTED HOMICIDE", "CRIMINAL THREATS", "SEXUAL ASSAULT"
    ]),
    ("Part 1 - Property", [
        "BURGLARY", "GRAND THEFT AUTO", "MOTOR VEHICLE THEFT",
        "SHOPLIFT", "THEFT", "LARCENY", "STOLEN", "VANDALISM", "ARSON"
    ]),
    ("Part 2 - Statutory/Quality of Life", [
        "NARCOTIC", "DRUG", "COCAINE", "MARIJUANA", "PROSTITUTION",
        "DISORDERLY", "PUBLIC INTOX", "DRUNK", "DUI", "TRESPASS",
        "WEAPONS OFFENSE"
    ])
]

df["Crime Category (Analytical)"] = categorize(
    df["Crm Cd Desc"], crime_rules,
    "Part 2 - Other / Miscellaneous"
)


# Weapon categories
weapon_rules = [
    ("Firearm", [
        "HANDGUN", "HAND GUN", "PISTOL", "RIFLE", "SHOTGUN",
        "REVOLVER", "FIREARM", "GUN"
    ]),
    ("Edged Weapon", [
        "KNIFE", "BLADE", "MACHETE", "RAZOR", "SWORD", "SCISSORS"
    ]),
    ("Blunt Object", [
        "BAT", "HAMMER", "CLUB", "STICK", "ROCK", "BRICK",
        "PIPE", "BOTTLE", "BOARD"
    ]),
    ("Bodily Force / Threat", [
        "STRONG-ARM", "STRONG ARM", "HANDS", "FIST", "BODILY FORCE",
        "VERBAL THREAT", "THREAT", "CHOKE", "STRANGULATION"
    ]),
    ("Chemical / Special Weapon", [
        "PEPPER SPRAY", "TASER", "STUN GUN", "TEAR GAS",
        "CAUSTIC", "ACID", "SPECIAL WEAPON"
    ])
]

df["Weapon Category"] = categorize(
    df["Weapon Desc"], weapon_rules,
    "Other / Unknown Weapon",
    missing="No Weapon Recorded / Missing"
)


# Status labels
df["Status Desc"] = df["Status Desc"].map({
    "Invest Cont": "Open / Investigation Continued",
    "Adult Arrest": "Cleared (Adult Arrest)",
    "Adult Other": "Cleared (Adult Other)",
    "Juv Arrest": "Cleared (Juvenile Arrest)",
    "Juv Other": "Cleared (Juvenile Other)"
}).fillna("Unknown")


# Display results
for col in ["Crime Category (Analytical)", "Weapon Category", "Status Desc"]:
    print(f"\n{col} Distribution:")
    display(df[col].value_counts())

print("Categorization complete!")

In [ ]:
# Select columns
crime_time = df[[
    "year", "month", "month_name", "Crm Cd Desc",
    "AREA NAME", "hour", "day_of_week", "day_name"
]].copy()

# Fill missing values
crime_time[["Crm Cd Desc", "AREA NAME", "month_name", "day_name"]] = (
    crime_time[["Crm Cd Desc", "AREA NAME", "month_name", "day_name"]]
    .fillna("Missing")
)

# Clean numeric columns
for col in ["hour", "day_of_week"]:
    crime_time[col] = (
        pd.to_numeric(crime_time[col], errors="coerce")
        .fillna(-1)
        .astype(int)
    )

# Classify time of day
hour = crime_time["hour"]

crime_time["time_of_day"] = np.select(
    [hour < 0, hour <= 5, hour <= 11, hour <= 17],
    ["Missing", "Night", "Morning", "Afternoon"],
    default="Evening"
)

# Display results
display(crime_time.head())

In [ ]:
# Group and count incidents
crime_time_detail = (
    crime_time
    .groupby([
        "year", "month", "month_name", "Crm Cd Desc",
        "AREA NAME", "hour", "time_of_day",
        "day_of_week", "day_name"
    ], dropna=False)
    .size()
    .reset_index(name="incident_count")
    .sort_values([
        "year", "month", "day_of_week", "hour",
        "Crm Cd Desc", "AREA NAME"
    ])
    .reset_index(drop=True)
)

# Save to Parquet
crime_time_output = SUMMARY_PATH / "crime_time_detail.parquet"
crime_time_detail.to_parquet(crime_time_output, index=False)

print("Shape:", crime_time_detail.shape)
print("Saved to:", crime_time_output)

In [ ]:
# Select columns
location_reporting = df[[
    "year", "Crm Cd Desc", "AREA NAME",
    "Premis Desc", "reporting_lag_days"
]].copy()

# Fill missing text values
text_cols = ["Crm Cd Desc", "AREA NAME", "Premis Desc"]
location_reporting[text_cols] = location_reporting[text_cols].fillna("Missing")

# Clean reporting lag
location_reporting["reporting_lag_days"] = (
    pd.to_numeric(location_reporting["reporting_lag_days"], errors="coerce")
    .fillna(-1)
    .round()
    .astype(int)
)

display(location_reporting.head())

In [ ]:
location_reporting_detail = (
    location_reporting
    .groupby([
        "year", "Crm Cd Desc", "AREA NAME",
        "Premis Desc", "reporting_lag_days"
    ], dropna=False)
    .size()
    .reset_index(name="incident_count")
    .sort_values([
        "year", "Crm Cd Desc", "AREA NAME",
        "Premis Desc", "reporting_lag_days"
    ])
    .reset_index(drop=True)
)

# Save to Parquet
location_output = SUMMARY_PATH / "location_reporting_detail.parquet"
location_reporting_detail.to_parquet(location_output, index=False)

print("Shape:", location_reporting_detail.shape)
print("Saved to:", location_output)

In [ ]:
# Select columns
crime_profile = df[[
    "year",
    "Crm Cd Desc",
    "Crime Category (Analytical)",
    "Status Desc",
    "Weapon Category"
]].copy()

# Fill missing values
crime_profile = crime_profile.fillna({
    "Crm Cd Desc": "Missing",
    "Crime Category (Analytical)": "Missing",
    "Status Desc": "Unknown",
    "Weapon Category": "No Weapon Recorded / Missing"
})

# Group, count, and sort incidents
crime_profile_detail = (
    crime_profile
    .groupby([
        "year",
        "Crm Cd Desc",
        "Crime Category (Analytical)",
        "Status Desc",
        "Weapon Category"
    ], dropna=False)
    .size()
    .reset_index(name="incident_count")
    .sort_values([
        "year",
        "Crime Category (Analytical)",
        "Crm Cd Desc",
        "Status Desc",
        "Weapon Category"
    ])
    .reset_index(drop=True)
)

# Save results
crime_profile_output = SUMMARY_PATH / "crime_profile_detail.parquet"
crime_profile_detail.to_parquet(crime_profile_output, index=False)

print("Shape:", crime_profile_detail.shape)
print("Saved to:", crime_profile_output)

display(crime_profile_detail.head())

In [ ]:
# Reload Parquet files
files = {
    "Crime & Time": crime_time_output,
    "Location & Reporting": location_output,
    "Crime Profile": crime_profile_output,
}

checks = {
    name: pd.read_parquet(path, engine="pyarrow")
    for name, path in files.items()
}

# Validate incident counts
for name, data in checks.items():
    assert data["incident_count"].sum() == len(df), f"{name} count mismatch"

# Validate sentinel values
assert checks["Crime & Time"]["hour"].ge(-1).all()
assert checks["Crime & Time"]["day_of_week"].ge(-1).all()
assert checks["Location & Reporting"]["reporting_lag_days"].ge(-1).all()

# Validate Crime Profile columns
assert checks["Crime Profile"].columns.tolist() == [
    "year",
    "Crm Cd Desc",
    "Crime Category (Analytical)",
    "Status Desc",
    "Weapon Category",
    "incident_count",
]

# Display results
print("Detailed dashboard-summary validation passed.")

for name, data in checks.items():
    print(f"{name} rows: {len(data):,}")

print(f"Master rows represented: {len(df):,}")